In [1]:
import os
from pathlib import Path

def print_tree(root_path: str | Path, max_depth: int = None, show_size: bool = False):
    """
    打印目录树结构。

    Parameters
    ----------
    root_path : str or Path
        根目录路径。
    max_depth : int, optional
        最大递归层数（默认无限制）。
    show_size : bool, optional
        是否显示文件大小（单位：KB）。
    """
    root_path = Path(root_path).resolve()
    print(f"📁 {root_path}")

    def _walk(dir_path: Path, prefix: str = "", depth: int = 0):
        if max_depth is not None and depth >= max_depth:
            return
        entries = sorted(dir_path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        count = len(entries)
        for i, entry in enumerate(entries):
            connector = "└── " if i == count - 1 else "├── "
            size_str = ""
            if show_size and entry.is_file():
                size_kb = entry.stat().st_size / 1024
                size_str = f" ({size_kb:.1f} KB)"
            print(prefix + connector + (f"{entry.name}{size_str}" if entry.is_file() else f"📁 {entry.name}"))
            if entry.is_dir():
                extension = "    " if i == count - 1 else "│   "
                _walk(entry, prefix + extension, depth + 1)

    _walk(root_path)

# === 示例 ===
# 打印当前目录
print_tree("expanded", show_size=True)

# 或者指定目录 + 限制深度
# print_tree("/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO", max_depth=3)


📁 /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded
├── 📁 round001
│   ├── adv_train_mspec=wwwwwwwwww_norm2_alpha2.5_epsilon8_steps10.pt (1622019.8 KB)
│   ├── attack_round001_timing.csv (8.6 KB)
│   └── manifest.json (0.2 KB)
├── 📁 round002
│   ├── adv_train_mspec=wwwwwwwwww_norm2_alpha5_epsilon15_steps10.pt (2027524.1 KB)
│   ├── attack_round002_timing.csv (10.8 KB)
│   └── manifest.json (0.2 KB)
├── 📁 round003
│   ├── adv_train_mspec=wwwwwwwwww_norm2_alpha10_epsilon35_steps10.pt (2128900.2 KB)
│   ├── attack_round003_timing.csv (11.2 KB)
│   └── manifest.json (0.2 KB)
├── 📁 round004
│   ├── adv_train_mspec=wwwwwwwwww_norm2_alpha50_epsilon105_steps10.pt (1622019.9 KB)
│   ├── attack_round004_timing.csv (7.9 KB)
│   └── manifest.json (0.2 KB)
├── 📁 round005
│   ├── adv_train_mspec=wwwwwwwwww_norm2_alpha50_epsilon105_steps10.pt (2027524.2 KB)
│   ├── attack_round005_timing.csv (10.2 KB)
│   └── manifest.json (0.2 K

In [2]:
import os
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

def plot_10_frames_grid(arr_4d, out_path, title="", cmap="viridis", figsize_per_cell=2.2, dpi=150):
    """
    arr_4d: (H, W, T) with T>=10
    画 4x5 网格 (t=0..9)，统一色标，保存到 out_path
    """
    assert arr_4d.ndim == 3, f"expected (H,W,T), got {arr_4d.shape}"
    H, W, T = arr_4d.shape
    assert T >= 10, f"T must be >=10, got {T}"

    # 统一色标
    vmin = float(arr_4d[:, :, :10].min())
    vmax = float(arr_4d[:, :, :10].max())

    rows, cols = 4, 5
    fig_w, fig_h = cols * figsize_per_cell, rows * figsize_per_cell
    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), constrained_layout=True)

    ims = []
    for t in range(10):
        r, c = divmod(t, cols)
        ax = axes[r, c]
        im = ax.imshow(arr_4d[:, :, t], cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
        ims.append(im)
        ax.set_title(f"t={t}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

    # 全局 colorbar
    cbar = fig.colorbar(ims[0], ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
    cbar.ax.set_ylabel("value", rotation=270, labelpad=12)

    if title:
        fig.suptitle(title, fontsize=11)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)

def process_rounds(
    root="/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded",
    pattern="round*/adv_train_*.pt",
    out_root="outputs_png",
    max_samples=5,
    cmap="viridis",
    dpi=150,
    figsize_per_cell=2.2,
):
    """
    遍历每个 round 的 .pt 数据，取前 max_samples 条样本，
    分别画 x_adv 与 y 的 10 帧 (4x5) 网格，并分类保存。
    """
    root = Path(root)
    out_root = Path(out_root)
    files = sorted(root.glob(pattern))

    if not files:
        print(f"[warn] no files matched: {root}/{pattern}")
        return []

    saved = []
    for f in files:
        try:
            data = torch.load(f, weights_only=False)
        except Exception as e:
            print(f"[skip] load failed: {f} | {e}")
            continue

        # 取键：优先 x_adv；若无则用 x
        if "x_adv" in data:
            X = data["x_adv"]
        elif "x" in data:
            X = data["x"]
        else:
            print(f"[skip] neither 'x_adv' nor 'x' in {f}")
            continue

        if "y" not in data:
            print(f"[skip] key 'y' not in {f}")
            continue

        # 形状检查
        if not (isinstance(X, torch.Tensor) and X.ndim == 4 and isinstance(data["y"], torch.Tensor) and data["y"].ndim == 4):
            print(f"[skip] unexpected shapes in {f}: x={getattr(X,'shape',None)}, y={getattr(data['y'],'shape',None)}")
            continue

        Bx, Hx, Wx, Tx = X.shape
        By, Hy, Wy, Ty = data["y"].shape
        if min(Bx, By) < 1 or min(Tx, Ty) < 10:
            print(f"[skip] need B>=1 and T>=10 in {f}, got x={X.shape}, y={data['y'].shape}")
            continue

        B = min(Bx, By, max_samples)  # 取前 B 条
        round_dir = f.parent.name  # e.g., round005
        stem = f.stem  # 文件名不含后缀

        for b in range(B):
            # x_adv / x
            xa = X[b].detach().cpu().numpy()          # (H,W,T)
            ya = data["y"][b].detach().cpu().numpy()  # (H,W,T)

            # x_adv 图
            out_x = out_root / "x_adv" / round_dir / f"{stem}_b{b}_xadv_t0to9.png"
            plot_10_frames_grid(
                xa, out_x,
                title=f"{round_dir} | {stem} | sample b={b} | x_adv (or x)",
                cmap=cmap, dpi=dpi, figsize_per_cell=figsize_per_cell
            )
            saved.append(str(out_x))

            # y 图
            out_y = out_root / "y" / round_dir / f"{stem}_b{b}_y_t0to9.png"
            plot_10_frames_grid(
                ya, out_y,
                title=f"{round_dir} | {stem} | sample b={b} | y",
                cmap=cmap, dpi=dpi, figsize_per_cell=figsize_per_cell
            )
            saved.append(str(out_y))

            print(f"[saved] {out_x}")
            print(f"[saved] {out_y}")

    return saved

# 运行
saved_paths = process_rounds(
    root="/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded",
    pattern="round*/adv_train_*.pt",
    out_root="outputs_png",
    max_samples=5,
)


[skip] unexpected shapes in /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded/round001/adv_train_mspec=wwwwwwwwww_norm2_alpha2.5_epsilon8_steps10.pt: x=torch.Size([288, 256, 256]), y=torch.Size([288, 256, 256, 21])
[skip] unexpected shapes in /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded/round002/adv_train_mspec=wwwwwwwwww_norm2_alpha5_epsilon15_steps10.pt: x=torch.Size([360, 256, 256]), y=torch.Size([360, 256, 256, 21])
[skip] unexpected shapes in /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded/round003/adv_train_mspec=wwwwwwwwww_norm2_alpha10_epsilon35_steps10.pt: x=torch.Size([378, 256, 256]), y=torch.Size([378, 256, 256, 21])


KeyboardInterrupt: 

In [3]:
import os
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

def plot_21_frames_grid(arr_4d, out_path, title="", cmap="viridis", figsize_per_cell=2.2, dpi=150):
    """
    arr_4d: (H, W, T) 且 T==21
    画 3x7 网格 (t=0..20)，统一色标，保存到 out_path
    """
    assert arr_4d.ndim == 3, f"expected (H,W,T), got {arr_4d.shape}"
    H, W, T = arr_4d.shape
    assert T == 21, f"T must be 21, got {T}"

    # 统一色标（对这个样本的全部 21 帧）
    vmin = float(arr_4d.min())
    vmax = float(arr_4d.max())

    rows, cols = 3, 7
    fig_w, fig_h = cols * figsize_per_cell, rows * figsize_per_cell
    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), constrained_layout=True)

    ims = []
    for t in range(21):
        r, c = divmod(t, cols)
        ax = axes[r, c]
        im = ax.imshow(arr_4d[:, :, t], cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
        ims.append(im)
        ax.set_title(f"t={t}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

    # 全局 colorbar
    cbar = fig.colorbar(ims[0], ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
    cbar.ax.set_ylabel("value", rotation=270, labelpad=12)

    if title:
        fig.suptitle(title, fontsize=11)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)

def process_rounds_y_only(
    root="/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded",
    pattern="round*/adv_train_*.pt",
    out_root="outputs_png",
    max_samples=5,
    cmap="viridis",
    dpi=150,
    figsize_per_cell=2.2,
):
    """
    遍历每个 round 的 .pt 数据，只画 y：
      - 从每个文件取前 max_samples 条样本（不足则全取）
      - 对每个样本画 3x7 网格（21 帧）
      - 保存到 outputs_png/y/roundXXX/
    """
    root = Path(root)
    out_root = Path(out_root)
    files = sorted(root.glob(pattern))

    if not files:
        print(f"[warn] no files matched: {root}/{pattern}")
        return []

    saved = []
    for f in files:
        try:
            data = torch.load(f, weights_only=False)
        except Exception as e:
            print(f"[skip] load failed: {f} | {e}")
            continue

        if "y" not in data:
            print(f"[skip] key 'y' not in {f}")
            continue

        y = data["y"]
        if not (isinstance(y, torch.Tensor) and y.ndim == 4 and y.shape[-1] == 21):
            print(f"[skip] unexpected y shape in {f}: {getattr(y,'shape',None)} (expect (N,H,W,21))")
            continue

        N, H, W, T = y.shape
        B = min(N, max_samples)
        round_dir = f.parent.name  # e.g., round005
        stem = f.stem

        for b in range(B):
            ya = y[b].detach().cpu().numpy()  # (H,W,21)
            out_y = out_root / "y" / round_dir / f"{stem}_b{b}_y_t0to20.png"
            plot_21_frames_grid(
                ya, out_y,
                title=f"{round_dir} | {stem} | sample b={b} | y (21 frames)",
                cmap=cmap, dpi=dpi, figsize_per_cell=figsize_per_cell
            )
            print(f"[saved] {out_y}")
            saved.append(str(out_y))

    return saved

# 运行示例：
saved_paths = process_rounds_y_only(
    root="/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/continual_train_attack/outputs/expanded",
    pattern="round*/adv_train_*.pt",
    out_root="outputs_png",
    max_samples=5,
)


[saved] outputs_png/y/round001/adv_train_mspec=wwwwwwwwww_norm2_alpha2.5_epsilon8_steps10_b0_y_t0to20.png
[saved] outputs_png/y/round001/adv_train_mspec=wwwwwwwwww_norm2_alpha2.5_epsilon8_steps10_b1_y_t0to20.png
[saved] outputs_png/y/round001/adv_train_mspec=wwwwwwwwww_norm2_alpha2.5_epsilon8_steps10_b2_y_t0to20.png
[saved] outputs_png/y/round001/adv_train_mspec=wwwwwwwwww_norm2_alpha2.5_epsilon8_steps10_b3_y_t0to20.png
[saved] outputs_png/y/round001/adv_train_mspec=wwwwwwwwww_norm2_alpha2.5_epsilon8_steps10_b4_y_t0to20.png
[saved] outputs_png/y/round002/adv_train_mspec=wwwwwwwwww_norm2_alpha5_epsilon15_steps10_b0_y_t0to20.png
[saved] outputs_png/y/round002/adv_train_mspec=wwwwwwwwww_norm2_alpha5_epsilon15_steps10_b1_y_t0to20.png
[saved] outputs_png/y/round002/adv_train_mspec=wwwwwwwwww_norm2_alpha5_epsilon15_steps10_b2_y_t0to20.png
[saved] outputs_png/y/round002/adv_train_mspec=wwwwwwwwww_norm2_alpha5_epsilon15_steps10_b3_y_t0to20.png
[saved] outputs_png/y/round002/adv_train_mspec=www